# Eyeblink8: independent recomputation of the D3 false-trigger result

**Claim being checked** (MorsEye paper, §5, Table 1): on Eyeblink8's natural blinks, a 200 ms reject window cuts
false triggers per minute with the blendshape detector D3 from **10.5 [7.5, 13.2]** to **4.1 [2.7, 5.7]**
(mean over 8 videos, 95% percentile bootstrap CI).

This notebook is written from scratch from the method as *described* in the paper, PRD and preregistration.
No code from the MorsEye repository is used or imported.

**What comes from where**
* **Dataset annotations** (`.tag`, `.txt`): frame timestamps, video length in minutes, the 408 annotated blinks.
* **Detector signal**: the annotations contain no blink scores, so D3's signal is recomputed here by running
  MediaPipe FaceLandmarker (Python `mediapipe`, model `face_landmarker.task`) on every frame of each `.avi`.

**Method as described in the paper**
* Signal: `max(eyeBlinkLeft, eyeBlinkRight)` per frame.
* D3: closed when the signal reaches the fixed threshold 0.5; hysteresis so it reopens only below `0.5 - h`.
  Here `h = 0.1`, i.e. reopen below 0.4. The paper defines h only for calibrated detectors (10% of the
  open-closed range); for the fixed detector this is read as 10% of the 0-1 score range. It is the value that
  reproduces the paper's per-video rates and duration errors (section 6 shows h = 0, 0.05 and 0.1).
* Durations from frame timestamps: first closed frame to first open frame.
* Face lost during a closure: the partial closure is discarded (PRD §4, `FACE_LOST`).
* Every blink in Eyeblink8 is spontaneous, so every closure the segmenter does not reject is a false trigger.
  Without the window every detected closure counts; with it, only closures of at least 200 ms.
* Per-video rate = triggers / video minutes; summary = mean over videos with a 95% percentile bootstrap
  (10,000 resamples, seed 2026).

Known source of small differences: the paper extracted features with the browser `@mediapipe/tasks-vision` build;
this notebook uses the Python `mediapipe` package, whose scores can differ slightly near the threshold.

In [1]:
import os, glob, time
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
from scipy.stats import wilcoxon

DATASET   = r"D:\datasets\eyeblink8\eyeblink8"
MODEL     = r"C:\Users\Saksham\face_landmarker.task"
CACHE_DIR = r"C:\Users\Saksham\eyeblink8_features"   # per-frame blink scores, so re-runs skip MediaPipe

THRESHOLD  = 0.5     # D3 fixed threshold
HYST       = 0.1     # reopen below THRESHOLD - HYST
REJECT_MS  = 200.0   # reject window
TXT_OFFSET = 0       # decoded frame i  <->  .txt row i + TXT_OFFSET  (see section 2)
N_BOOT, SEED = 10_000, 2026

# Paper, Table 1 (D3 FT/min without -> with the reject window), for comparison only
PAPER = {"1": (3.5, 1.7), "2": (13.7, 8.0), "3": (9.0, 3.3), "4": (9.3, 3.3),
         "8": (5.2, 1.5), "9": (15.1, 5.3), "10": (13.3, 3.4), "11": (14.7, 6.6)}
PAPER_BLINKS = {"1": 38, "2": 88, "3": 65, "4": 31, "8": 30, "9": 41, "10": 72, "11": 43}

os.makedirs(CACHE_DIR, exist_ok=True)
VIDEOS = sorted(os.listdir(DATASET), key=int)
print("mediapipe", mp.__version__, "| videos:", VIDEOS)

mediapipe 1.2.0 | videos: ['1', '2', '3', '4', '8', '9', '10', '11']


## 1. Annotations: timestamps and annotated blinks

`.txt`: one line per recorded frame, `frame_index timestamp_seconds`.
`.tag`: after `#start`, one line per annotated frame, `frame:blinkID:NF:LE_FC:LE_NV:RE_FC:RE_NV:...`,
with `blinkID = -1` outside blinks. A blink's annotated interval runs from its first frame to the frame after
its last frame. Blinks are kept as frame indices and turned into times once the frame-to-timestamp mapping
is fixed (section 2).

In [2]:
def paths(v):
    stem = glob.glob(os.path.join(DATASET, v, "*.avi"))[0][:-4]
    return stem + ".avi", stem + ".tag", stem + ".txt"

def load_timestamps(txt):
    rows = [l.split() for l in open(txt) if l.strip()]
    idx = np.array([int(r[0]) for r in rows])
    t = np.array([float(r[1]) for r in rows])
    assert (idx == np.arange(len(idx))).all(), "frame indices not contiguous"
    return t                                   # seconds, index = .txt row

def load_blink_frames(tag):
    body = open(tag).read().split("#start", 1)[1]
    blinks, last_frame = {}, 0
    for line in body.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        f = line.split(":")
        frame, bid = int(f[0]), int(f[1])
        last_frame = max(last_frame, frame)
        if bid >= 0:
            blinks.setdefault(bid, []).append(frame)
    iv = np.array([(min(fr), max(fr) + 1) for _, fr in sorted(blinks.items())])
    return iv, last_frame                      # (n, 2) [first frame, frame after last), last annotated frame

ANN = {}
for v in VIDEOS:
    avi, tag, txt = paths(v)
    t_all = load_timestamps(txt)
    iv, last = load_blink_frames(tag)
    ANN[v] = dict(t_all=t_all, blink_frames=iv, last_tag_frame=last)
    print(f"video {v:>2}: {len(t_all):6d} .txt rows, {len(iv):3d} blinks (paper {PAPER_BLINKS[v]}), "
          f"last annotated frame {last}")
print("total blinks:", sum(len(a["blink_frames"]) for a in ANN.values()), "(paper 408)")

video  1:  15833 .txt rows,  38 blinks (paper 38), last annotated frame 15782
video  2:  11231 .txt rows,  88 blinks (paper 88), last annotated frame 11181
video  3:   9265 .txt rows,  65 blinks (paper 65), last annotated frame 9215
video  4:   5454 .txt rows,  31 blinks (paper 31), last annotated frame 5403
video  8:  10712 .txt rows,  30 blinks (paper 30), last annotated frame 10662
video  9:   5183 .txt rows,  41 blinks (paper 41), last annotated frame 5133
video 10:   9126 .txt rows,  72 blinks (paper 72), last annotated frame 9076
video 11:   4944 .txt rows,  43 blinks (paper 43), last annotated frame 4894
total blinks: 408 (paper 408)


## 2. Blink scores from the videos (MediaPipe FaceLandmarker)

**Frame count.** Every `.avi` decodes to exactly 49 fewer frames than its `.txt` lists (checked with both
OpenCV and PyAV, which agree pixel for pixel). The `.tag` frame numbers run from 0 up to the decoded count minus 1,
so the annotations index the decoded frames. Following the dataset's convention, decoded frame *i* gets `.txt`
row *i* (`TXT_OFFSET = 0`) and the last 49 `.txt` rows are unused. Section 6 repeats the headline with the other
plausible alignment (`TXT_OFFSET = 49`).

VIDEO running mode, one face; MediaPipe gets each frame's `.txt` timestamp. Frames with no face are NaN.
Results are cached in `CACHE_DIR`; delete it to re-extract. The first run takes a while on a CPU.

In [3]:
def make_landmarker():
    opts = vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL),
        running_mode=vision.RunningMode.VIDEO, num_faces=1, output_face_blendshapes=True)
    return vision.FaceLandmarker.create_from_options(opts)

def extract_scores(v):
    cache = os.path.join(CACHE_DIR, f"video_{v}.npz")
    if os.path.exists(cache):
        return np.load(cache)["score"]
    avi, _, _ = paths(v)
    t_all = ANN[v]["t_all"]
    lm, cap = make_landmarker(), cv2.VideoCapture(avi)
    scores, last_ms, t0 = [], -1, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        i = len(scores)
        ms = max(int(round(t_all[i] * 1000)), last_ms + 1)    # must strictly increase
        last_ms = ms
        res = lm.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,
                                           data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), ms)
        if res.face_blendshapes:
            bs = {c.category_name: c.score for c in res.face_blendshapes[0]}
            scores.append(max(bs["eyeBlinkLeft"], bs["eyeBlinkRight"]))
        else:
            scores.append(np.nan)
    cap.release()
    lm.close()
    score = np.array(scores)
    np.savez_compressed(cache, score=score)
    print(f"video {v}: extracted {len(score)} frames in {time.time() - t0:.0f} s")
    return score

for v in VIDEOS:
    a = ANN[v]
    a["score"] = extract_scores(v)
    n = len(a["score"])
    assert a["last_tag_frame"] < n, "annotations reference a frame that does not decode"
    print(f"video {v:>2}: decoded {n} frames ({len(a['t_all']) - n} fewer than .txt), "
          f"no face in {np.isnan(a['score']).mean():.2%}")

def timebase(v, offset=TXT_OFFSET):
    # Timestamps (s) for the decoded frames under a given frame-to-.txt alignment.
    a = ANN[v]
    return a["t_all"][offset: offset + len(a["score"])]

def minutes(v, offset=TXT_OFFSET):
    t = timebase(v, offset)
    return (t[-1] - t[0]) / 60

def blink_times(v, offset=TXT_OFFSET):
    t = timebase(v, offset)
    fr = np.minimum(ANN[v]["blink_frames"], len(t) - 1)
    return t[fr]                               # (n, 2) onset/offset in seconds

video  1: decoded 15784 frames (49 fewer than .txt), no face in 0.00%
video  2: decoded 11182 frames (49 fewer than .txt), no face in 0.53%
video  3: decoded 9216 frames (49 fewer than .txt), no face in 0.01%
video  4: decoded 5405 frames (49 fewer than .txt), no face in 0.83%
video  8: decoded 10663 frames (49 fewer than .txt), no face in 0.00%
video  9: decoded 5134 frames (49 fewer than .txt), no face in 0.29%
video 10: decoded 9077 frames (49 fewer than .txt), no face in 0.53%
video 11: decoded 4895 frames (49 fewer than .txt), no face in 0.00%


In [4]:
for v in VIDEOS:
    t = timebase(v)
    print(f"video {v:>2}: {minutes(v):5.2f} min, {1 / np.median(np.diff(t)):5.1f} fps (median)")
all_dur = np.concatenate([np.diff(blink_times(v), axis=1)[:, 0] * 1000 for v in VIDEOS])
print(f"\nannotated blinks: median {np.median(all_dur):.0f} ms, "
      f"5-95th pct {np.percentile(all_dur, 5):.0f}-{np.percentile(all_dur, 95):.0f} ms, "
      f"{(all_dur < REJECT_MS).mean():.0%} under 200 ms   (paper: 272 ms, 192-560 ms, 9%)")

video  1:  8.88 min,  31.2 fps (median)
video  2:  6.37 min,  31.2 fps (median)
video  3:  5.20 min,  31.2 fps (median)
video  4:  3.00 min,  31.2 fps (median)
video  8:  5.93 min,  31.2 fps (median)
video  9:  2.85 min,  31.2 fps (median)
video 10:  5.04 min,  31.2 fps (median)
video 11:  2.72 min,  31.2 fps (median)

annotated blinks: median 272 ms, 5-95th pct 192-560 ms, 9% under 200 ms   (paper: 272 ms, 192-560 ms, 9%)


## 3. Detector D3 (fixed threshold + hysteresis) and false triggers

In [5]:
def detect_closures(t, score, thr=THRESHOLD, h=HYST):
    # (onset_s, offset_s) for each complete closure.
    # Closed when score >= thr; reopen at the first frame with score < thr - h.
    # A frame with no face discards any closure in progress.
    out, start = [], None
    for i, s in enumerate(score):
        if np.isnan(s):
            start = None
            continue
        if start is None:
            if s >= thr:
                start = i
        elif s < thr - h:
            out.append((t[start], t[i]))
            start = None
    return np.array(out).reshape(-1, 2)        # a closure still open at the end of the video is dropped

def false_triggers(v, thr=THRESHOLD, h=HYST, reject_ms=REJECT_MS, offset=TXT_OFFSET):
    cl = detect_closures(timebase(v, offset), ANN[v]["score"], thr, h)
    dur_ms = (cl[:, 1] - cl[:, 0]) * 1000
    m = minutes(v, offset)
    return len(cl) / m, (dur_ms >= reject_ms).sum() / m, cl

rows = []
print(f"{'video':>5} {'min':>5} {'closures':>8} | {'FT/min none':>11} {'FT/min 200ms':>12} | paper")
for v in VIDEOS:
    no_win, win, cl = false_triggers(v)
    rows.append((no_win, win))
    print(f"{v:>5} {minutes(v):5.1f} {len(cl):8d} | {no_win:11.1f} {win:12.1f} | "
          f"{PAPER[v][0]:.1f} -> {PAPER[v][1]:.1f}")
FT = np.array(rows)

video   min closures | FT/min none FT/min 200ms | paper
    1   8.9       31 |         3.5          1.7 | 3.5 -> 1.7
    2   6.4       87 |        13.7          8.2 | 13.7 -> 8.0
    3   5.2       47 |         9.0          3.3 | 9.0 -> 3.3
    4   3.0       28 |         9.3          3.3 | 9.3 -> 3.3
    8   5.9       31 |         5.2          1.5 | 5.2 -> 1.5
    9   2.9       44 |        15.4          5.3 | 15.1 -> 5.3


   10   5.0       68 |        13.5          3.4 | 13.3 -> 3.4
   11   2.7       40 |        14.7          6.6 | 14.7 -> 6.6


## 4. Mean over videos with 95% bootstrap CIs

In [6]:
def boot_ci(x, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), size=(n, len(x)))].mean(axis=1)
    return x.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

m0, lo0, hi0 = boot_ci(FT[:, 0])
m1, lo1, hi1 = boot_ci(FT[:, 1])
p = wilcoxon(FT[:, 0], FT[:, 1]).pvalue
print(f"D3 without window : {m0:4.1f} [{lo0:.1f}, {hi0:.1f}] FT/min   (paper 10.5 [7.5, 13.2])")
print(f"D3 with 200 ms    : {m1:4.1f} [{lo1:.1f}, {hi1:.1f}] FT/min   (paper  4.1 [2.7, 5.7])")
print(f"ratio of means    : {m1 / m0:.2f}                     (paper 0.39)")
print(f"fell in {int((FT[:, 1] < FT[:, 0]).sum())}/8 videos; Wilcoxon signed-rank p = {p:.3f} (paper 0.008)")
print("halving criterion met:", m1 <= m0 / 2)

D3 without window : 10.5 [7.5, 13.3] FT/min   (paper 10.5 [7.5, 13.2])
D3 with 200 ms    :  4.2 [2.7, 5.8] FT/min   (paper  4.1 [2.7, 5.7])
ratio of means    : 0.39                     (paper 0.39)
fell in 8/8 videos; Wilcoxon signed-rank p = 0.008 (paper 0.008)
halving criterion met: True


## 5. Check: measured vs annotated blink length

The paper explains the D3 result by D3 measuring blinks as short: for matched blinks, a median of -96 ms
(IQR -128 to -64) relative to the annotation, with 60% of D3's measurements under 200 ms (n = 354 matched).
Here a closure is matched to an annotated blink if their intervals overlap (first unused overlapping closure).

In [7]:
diffs, meas = [], []
for v in VIDEOS:
    cl = detect_closures(timebase(v), ANN[v]["score"])
    used = set()
    for a0, a1 in blink_times(v):
        hit = [j for j, (c0, c1) in enumerate(cl) if c0 < a1 and c1 > a0 and j not in used]
        if hit:
            j = hit[0]
            used.add(j)
            d = (cl[j, 1] - cl[j, 0]) * 1000
            meas.append(d)
            diffs.append(d - (a1 - a0) * 1000)
diffs, meas = np.array(diffs), np.array(meas)
print(f"matched blinks: {len(diffs)} of {len(all_dur)}   (paper n = 354)")
print(f"measured - annotated: median {np.median(diffs):.0f} ms, IQR {np.percentile(diffs, 25):.0f} to "
      f"{np.percentile(diffs, 75):.0f} ms   (paper -96, -128 to -64)")
print(f"measured closures under 200 ms: {(meas < REJECT_MS).mean():.0%}   (paper 60%)")

matched blinks: 362 of 408   (paper n = 354)


measured - annotated: median -96 ms, IQR -128 to -64 ms   (paper -96, -128 to -64)
measured closures under 200 ms: 60%   (paper 60%)


## 6. Sensitivity to the two assumptions

* `h`: not stated for the fixed detectors; `h = 0.1` is the value used for the result.
* Frame alignment: decoded frame *i* to `.txt` row *i* (used) or row *i* + 49.

In [8]:
for off in (0, 49):
    for h in (0.0, 0.05, 0.1):
        ft = np.array([false_triggers(v, h=h, offset=off)[:2] for v in VIDEOS])
        tag = "  <- reported" if (off, h) == (TXT_OFFSET, HYST) else ""
        print(f"offset {off:2d}, h = {h:4.2f}: {ft[:, 0].mean():4.1f} -> {ft[:, 1].mean():4.1f} FT/min{tag}")

offset  0, h = 0.00: 10.7 ->  2.8 FT/min


offset  0, h = 0.05: 10.6 ->  3.3 FT/min


offset  0, h = 0.10: 10.5 ->  4.2 FT/min  <- reported


offset 49, h = 0.00: 10.7 ->  2.8 FT/min
offset 49, h = 0.05: 10.6 ->  3.2 FT/min


offset 49, h = 0.10: 10.5 ->  4.0 FT/min
